# Lab 6: SageMaker Pipelines로 Bank Marketing ML 워크플로우 자동화

노트북 1~4의 **전처리 → 훈련 → 평가** 작업을 SageMaker Pipelines로 자동화합니다.

**파이프라인 구성:**

```
BankPreprocess
    ├─ BankTrainConservative  (병렬)
    └─ BankTrainAggressive    (병렬)
              └─ BankEvaluate
                      └─ BankAUCCondition
                              ├─ (AUC ≥ 임계값) BankRegisterModel
                              └─ (AUC <  임계값) BankPipelineFail
```

> ⚠️ **사전 조건**: `0-setup.ipynb`와 `1-preprocessing.ipynb`를 먼저 실행하여 환경 변수를 저장해야 합니다.

In [ ]:
# 패키지 설치 확인 (무한 루프 방지 가드)
import importlib.util, sys

if importlib.util.find_spec("sagemaker.sklearn") is None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "setuptools<81", "sagemaker>=2.220,<3", "mlflow==2.13.2", "sagemaker-mlflow"])
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import boto3
import sagemaker
import mlflow
import os

from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.parameters import ParameterString, ParameterFloat, ParameterInteger
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import Join, JsonGet
from sagemaker.workflow.properties import PropertyFile
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn import SKLearn
from sagemaker.processing import FrameworkProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker import image_uris
from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep

print(f"SageMaker SDK: {sagemaker.__version__}")

## 1. 환경 변수 복원

In [ ]:
%store -r bucket
%store -r prefix
%store -r role
%store -r region
%store -r mlflow_arn
%store -r experiment_name
%store -r input_source

# 필수 변수 확인
required = {"bucket": bucket, "prefix": prefix, "role": role,
            "region": region, "mlflow_arn": mlflow_arn,
            "experiment_name": experiment_name, "input_source": input_source}
missing = [k for k, v in required.items() if not v]
if missing:
    raise RuntimeError(f"누락된 변수: {missing}\n0-setup.ipynb를 먼저 실행하세요.")

print(f"bucket          : {bucket}")
print(f"prefix          : {prefix}")
print(f"region          : {region}")
print(f"mlflow_arn      : {mlflow_arn}")
print(f"experiment_name : {experiment_name}")
print(f"input_source    : {input_source}")

In [ ]:
boto_session      = boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)
pipeline_session  = PipelineSession(boto_session=boto_session)
sm_client         = boto_session.client("sagemaker")

print("세션 초기화 완료")

## 2. 파이프라인 파라미터

In [ ]:
processing_instance_type = ParameterString(
    name="ProcessingInstanceType", default_value="ml.m5.large")

training_instance_type = ParameterString(
    name="TrainingInstanceType", default_value="ml.m5.large")

model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval")

auc_threshold = ParameterFloat(
    name="AucThreshold", default_value=0.75)

input_data = ParameterString(
    name="InputData", default_value=input_source)

mlflow_tracking_arn = ParameterString(
    name="MlflowTrackingArn", default_value=mlflow_arn)

mlflow_experiment = ParameterString(
    name="MlflowExperimentName", default_value=experiment_name)

print("파이프라인 파라미터 정의 완료")

## 3. 스크립트 작성

In [ ]:
os.makedirs("pipeline_code", exist_ok=True)

In [ ]:
%%writefile pipeline_code/preprocessing.py
"""Bank Marketing 데이터 전처리 스크립트 (노트북 1과 동일 로직)"""
import argparse
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

if __name__ == "__main__":
    base_dir = "/opt/ml/processing"

    df = pd.read_csv(f"{base_dir}/input/bank-additional-full.csv", sep=";")

    # 컬럼명 정리: '.' → '_'
    df.columns = [c.replace(".", "_") for c in df.columns]

    # 파생 변수 생성
    df["no_previous_contact"] = (df["pdays"] == 999).astype(int)
    df["not_working"] = df["job"].isin(["student", "retired", "unemployed"]).astype(int)

    # 불필요 컬럼 제거
    drop_cols = ["duration", "emp_var_rate", "cons_price_idx",
                 "cons_conf_idx", "euribor3m", "nr_employed"]
    df.drop(columns=drop_cols, inplace=True)

    # 원-핫 인코딩
    df = pd.get_dummies(df).astype(int)

    # 레이블 첫 컬럼으로 이동
    label_col = "y_yes"
    cols = [label_col] + [c for c in df.columns if c != label_col]
    df = df[cols]

    # stratify 분할 70/20/10
    X = df.drop(columns=[label_col])
    y = df[label_col]

    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=1/3, random_state=42, stratify=y_tmp)

    # train/validation: 레이블 첫 컬럼, headerless CSV
    train_df = pd.concat([y_train, X_train], axis=1)
    val_df   = pd.concat([y_val,   X_val],   axis=1)

    os.makedirs(f"{base_dir}/output/train",      exist_ok=True)
    os.makedirs(f"{base_dir}/output/validation", exist_ok=True)
    os.makedirs(f"{base_dir}/output/test",       exist_ok=True)
    os.makedirs(f"{base_dir}/output/baseline",   exist_ok=True)

    train_df.to_csv(f"{base_dir}/output/train/train.csv",           index=False, header=False)
    val_df.to_csv(  f"{base_dir}/output/validation/validation.csv", index=False, header=False)
    X_test.to_csv(  f"{base_dir}/output/test/test_x.csv",           index=False, header=False)
    y_test.to_csv(  f"{base_dir}/output/test/test_y.csv",           index=False, header=False)
    train_df.to_csv(f"{base_dir}/output/baseline/baseline.csv",     index=False, header=False)

    print(f"train      : {len(train_df):,}행")
    print(f"validation : {len(val_df):,}행")
    print(f"test       : {len(X_test):,}행")

In [ ]:
%%writefile pipeline_code/requirements.txt
pandas
numpy
scikit-learn

In [ ]:
%%writefile pipeline_code/evaluation.py
"""두 XGBoost 모델을 비교하고 평가 리포트를 생성합니다."""
import os, json, tarfile
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score

BASE = "/opt/ml/processing"


def load_booster(model_dir):
    """model.tar.gz에서 xgboost-model 부스터를 로드합니다."""
    tar_path = os.path.join(model_dir, "model.tar.gz")
    if os.path.exists(tar_path):
        with tarfile.open(tar_path) as t:
            t.extractall(model_dir)
    booster = xgb.Booster()
    booster.load_model(os.path.join(model_dir, "xgboost-model"))
    return booster


def main():
    # 테스트 데이터 로드
    test_x = pd.read_csv(f"{BASE}/test/test_x.csv", header=None)
    test_y = pd.read_csv(f"{BASE}/test/test_y.csv", header=None).squeeze()
    dtest  = xgb.DMatrix(test_x)

    # 두 모델 로드 및 예측
    model1 = load_booster(f"{BASE}/model1")
    model2 = load_booster(f"{BASE}/model2")

    pred1 = model1.predict(dtest)
    pred2 = model2.predict(dtest)

    # 메트릭 계산
    auc1 = float(roc_auc_score(test_y, pred1))
    auc2 = float(roc_auc_score(test_y, pred2))
    acc1 = float(accuracy_score(test_y, (pred1 > 0.5).astype(int)))
    acc2 = float(accuracy_score(test_y, (pred2 > 0.5).astype(int)))

    best_model_num = 1 if auc1 >= auc2 else 2
    best_auc       = max(auc1, auc2)

    print(f"[Model 1 - Conservative] AUC: {auc1:.4f}  Accuracy: {acc1:.4f}")
    print(f"[Model 2 - Aggressive  ] AUC: {auc2:.4f}  Accuracy: {acc2:.4f}")
    print(f"=> Best: Model {best_model_num}  (AUC: {best_auc:.4f})")

    # evaluation.json 작성 (PropertyFile로 노출)
    report = {
        "evaluation_metrics": {
            "model1_auc":   {"value": auc1, "standard_deviation": "NaN"},
            "model2_auc":   {"value": auc2, "standard_deviation": "NaN"},
            "model1_accuracy": {"value": acc1, "standard_deviation": "NaN"},
            "model2_accuracy": {"value": acc2, "standard_deviation": "NaN"},
            "best_auc":     {"value": best_auc, "standard_deviation": "NaN"},
            "best_model":   {"value": float(best_model_num), "standard_deviation": "NaN"}
        }
    }
    os.makedirs(f"{BASE}/evaluation", exist_ok=True)
    with open(f"{BASE}/evaluation/evaluation.json", "w") as f:
        json.dump(report, f, indent=2)
    print("evaluation.json 저장 완료")

    # MLflow 로깅 (선택적 — 실패해도 잡 종료 안 함)
    try:
        mlflow_tracking_arn = os.environ.get("MLFLOW_TRACKING_ARN")
        exp_name = os.environ.get("MLFLOW_EXPERIMENT_NAME")
        if mlflow_tracking_arn:
            import mlflow
            mlflow.set_tracking_uri(mlflow_tracking_arn)
            mlflow.set_experiment(exp_name or "bank-pipeline")
            with mlflow.start_run(run_name="pipeline-evaluation"):
                mlflow.log_metrics({
                    "pipeline_model1_auc":      auc1,
                    "pipeline_model2_auc":      auc2,
                    "pipeline_model1_accuracy": acc1,
                    "pipeline_model2_accuracy": acc2,
                    "pipeline_best_auc":        best_auc,
                    "pipeline_best_model":      float(best_model_num)
                })
                mlflow.set_tag("best_model", f"model{best_model_num}")
            print("MLflow 로깅 완료")
    except Exception as e:
        print(f"MLflow 로깅 스킵 (선택적): {e}")


if __name__ == "__main__":
    main()

## 4. 파이프라인 단계 정의

### Step 1: 데이터 전처리

In [ ]:
sklearn_processor = FrameworkProcessor(
    estimator_cls=SKLearn,
    framework_version="1.2-1",
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="bank-pipeline-preprocess",
    sagemaker_session=pipeline_session
)

processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input",
            s3_data_distribution_type="FullyReplicated"
        )
    ],
    outputs=[
        ProcessingOutput(output_name="train",      source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation"),
        ProcessingOutput(output_name="test",       source="/opt/ml/processing/output/test"),
        ProcessingOutput(output_name="baseline",   source="/opt/ml/processing/output/baseline"),
    ],
    code="pipeline_code/preprocessing.py",
    dependencies=["pipeline_code/requirements.txt"]
)

step_preprocess = ProcessingStep(name="BankPreprocess", step_args=processor_args)
print("BankPreprocess 단계 정의 완료")

### Step 2: 모델 훈련 (Conservative / Aggressive 병렬)

In [ ]:
xgb_image_uri = image_uris.retrieve(
    framework="xgboost", region=region,
    version="1.7-1", image_scope="training"
)

train_s3_uri = step_preprocess.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri
val_s3_uri   = step_preprocess.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri

# --- Model 1: Conservative ---
xgb_conservative = Estimator(
    image_uri=xgb_image_uri,
    instance_type=training_instance_type,
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/pipeline/output/model1/",
    role=role,
    base_job_name="bank-pipeline-conservative",
    sagemaker_session=pipeline_session
)
xgb_conservative.set_hyperparameters(
    max_depth=3, eta=0.5, gamma=4,
    min_child_weight=6, subsample=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    num_round=50, verbosity=0
)
train_args_1 = xgb_conservative.fit(
    inputs={
        "train":      TrainingInput(s3_data=train_s3_uri, content_type="text/csv"),
        "validation": TrainingInput(s3_data=val_s3_uri,   content_type="text/csv")
    }
)
step_train1 = TrainingStep(name="BankTrainConservative", step_args=train_args_1)

# --- Model 2: Aggressive ---
xgb_aggressive = Estimator(
    image_uri=xgb_image_uri,
    instance_type=training_instance_type,
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/pipeline/output/model2/",
    role=role,
    base_job_name="bank-pipeline-aggressive",
    sagemaker_session=pipeline_session
)
xgb_aggressive.set_hyperparameters(
    max_depth=3, eta=0.1, gamma=2,
    min_child_weight=3, subsample=0.4,
    objective="binary:logistic",
    eval_metric="auc",
    num_round=100, verbosity=0
)
train_args_2 = xgb_aggressive.fit(
    inputs={
        "train":      TrainingInput(s3_data=train_s3_uri, content_type="text/csv"),
        "validation": TrainingInput(s3_data=val_s3_uri,   content_type="text/csv")
    }
)
step_train2 = TrainingStep(name="BankTrainAggressive", step_args=train_args_2)

print("BankTrainConservative / BankTrainAggressive 단계 정의 완료")

### Step 3: 모델 평가 (두 모델 AUC 비교 + MLflow 로깅)

In [ ]:
eval_processor = ScriptProcessor(
    image_uri=xgb_image_uri,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="bank-pipeline-evaluate",
    role=role,
    env={
        "MLFLOW_TRACKING_ARN": mlflow_tracking_arn,
        "MLFLOW_EXPERIMENT_NAME": mlflow_experiment
    },
    sagemaker_session=pipeline_session
)

test_s3_uri = step_preprocess.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri

eval_args = eval_processor.run(
    inputs=[
        ProcessingInput(
            source=step_train1.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model1"
        ),
        ProcessingInput(
            source=step_train2.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model2"
        ),
        ProcessingInput(
            source=test_s3_uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"
        )
    ],
    code="pipeline_code/evaluation.py"
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

step_eval = ProcessingStep(
    name="BankEvaluate",
    step_args=eval_args,
    property_files=[evaluation_report]
)

print("BankEvaluate 단계 정의 완료")

### Step 4: 조건부 모델 등록 (AUC ≥ 임계값)

In [ ]:
from sagemaker.model_metrics import ModelMetrics, MetricsSource

model_package_group = "BankMarketingModelPackageGroup"

# 등록 대상: Conservative 모델 (AUC 조건 통과 시)
register_model = Model(
    image_uri=xgb_image_uri,
    model_data=step_train1.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session
)

eval_output_uri = step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=f"{eval_output_uri}/evaluation.json",
        content_type="application/json"
    )
)

register_args = register_model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large", "ml.m5.xlarge"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group,
    approval_status=model_approval_status,
    model_metrics=model_metrics
)
step_register = ModelStep(name="BankRegisterModel", step_args=register_args)

# 실패 단계
step_fail = FailStep(
    name="BankPipelineFail",
    error_message=Join(
        on=" ",
        values=["AUC 임계값 미달:", auc_threshold, "기준 통과 필요"]
    )
)

# 조건: best_auc >= auc_threshold
cond_auc = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="evaluation_metrics.best_auc.value"
    ),
    right=auc_threshold
)

step_condition = ConditionStep(
    name="BankAUCCondition",
    conditions=[cond_auc],
    if_steps=[step_register],
    else_steps=[step_fail]
)

print("BankAUCCondition / BankRegisterModel / BankPipelineFail 단계 정의 완료")

## 5. 파이프라인 생성 및 등록

In [ ]:
pipeline_name = "BankMarketingPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_type,
        training_instance_type,
        model_approval_status,
        auc_threshold,
        input_data,
        mlflow_tracking_arn,
        mlflow_experiment
    ],
    steps=[
        step_preprocess,
        step_train1,
        step_train2,
        step_eval,
        step_condition
    ],
    sagemaker_session=pipeline_session
)

pipeline.upsert(role_arn=role)
print(f"파이프라인 등록 완료: {pipeline_name}")

## 6. 파이프라인 실행

In [ ]:
execution = pipeline.start(
    parameters={
        "AucThreshold": 0.75,
        "ModelApprovalStatus": "PendingManualApproval"
    }
)

print(f"파이프라인 실행 ARN: {execution.arn}")
print("실행 완료까지 약 15~20분 소요됩니다...")

In [ ]:
# 실행 완료 대기 (완료 또는 실패 시까지)
execution.wait()
print("파이프라인 실행 완료")

## 7. 실행 결과 확인

In [ ]:
import pandas as pd

steps = execution.list_steps()
step_df = pd.DataFrame([
    {
        "단계": s["StepName"],
        "상태": s["StepStatus"],
        "시작": s.get("StartTime", ""),
        "종료": s.get("EndTime", "")
    }
    for s in steps
])
print(step_df.to_string(index=False))

In [ ]:
# 평가 결과 확인
import json
from sagemaker.s3 import S3Downloader

eval_step = next(
    (s for s in steps if s["StepName"] == "BankEvaluate"), None
)

if eval_step and eval_step["StepStatus"] == "Succeeded":
    output_s3 = eval_step["Metadata"]["ProcessingJob"]["Arn"]
    job_name  = output_s3.split("/")[-1]
    desc      = sm_client.describe_processing_job(ProcessingJobName=job_name)
    eval_uri  = next(
        o["S3Output"]["S3Uri"]
        for o in desc["ProcessingOutputConfig"]["Outputs"]
        if o["OutputName"] == "evaluation"
    )
    report = json.loads(S3Downloader.read_file(f"{eval_uri}/evaluation.json"))
    metrics = report["evaluation_metrics"]

    print("=" * 45)
    print(f"  Model 1 (Conservative) AUC : {metrics['model1_auc']['value']:.4f}")
    print(f"  Model 1 (Conservative) ACC : {metrics['model1_accuracy']['value']:.4f}")
    print(f"  Model 2 (Aggressive)   AUC : {metrics['model2_auc']['value']:.4f}")
    print(f"  Model 2 (Aggressive)   ACC : {metrics['model2_accuracy']['value']:.4f}")
    print("-" * 45)
    print(f"  Best AUC  : {metrics['best_auc']['value']:.4f}")
    print(f"  Best Model: Model {int(metrics['best_model']['value'])}")
    print("=" * 45)
else:
    print("평가 단계가 아직 완료되지 않았거나 실패했습니다.")

In [ ]:
# MLflow 실험 결과 확인
mlflow.set_tracking_uri(mlflow_arn)
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name(experiment_name)

if exp:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName = 'pipeline-evaluation'",
        order_by=["start_time DESC"],
        max_results=1
    )
    if runs:
        r = runs[0]
        print(f"Run ID   : {r.info.run_id}")
        print(f"Best AUC : {r.data.metrics.get('pipeline_best_auc', 'N/A'):.4f}")
        print(f"Best Model: Model {int(r.data.metrics.get('pipeline_best_model', 0))}")
    else:
        print("pipeline-evaluation run을 찾을 수 없습니다.")
else:
    print(f"실험 '{experiment_name}'을 찾을 수 없습니다.")

## 8. 리소스 정리 (선택)

파이프라인 실행이 완료된 후 불필요한 리소스를 정리합니다.

In [ ]:
DELETE_PIPELINE = False  # True로 변경 시 파이프라인 삭제

if DELETE_PIPELINE:
    sm_client.delete_pipeline(PipelineName=pipeline_name)
    print(f"파이프라인 삭제 완료: {pipeline_name}")
else:
    print("DELETE_PIPELINE = False — 파이프라인 유지")